# 01 - ISIC Archive — Exploratory Data Analysis

Metadata EDA for the ISIC Archive classification stage. Builds the 4-class target
(melanocytic/non-melanocytic × benign/malignant), inspects class balance and imbalance,
demographics, image dimensions, and lesion/patient redundancy, then saves the cleaned,
imputed, one-hot-encoded dataset for the downstream classifiers.

> The images referenced here are later segmented by the Mask R-CNN trained on HAM10000
> (see `02_ham10000.ipynb`) and then fed to the classification models.

## Import dependencies

In [ ]:
import os
import sys

# Add the repo root (the directory that contains `src/`) to sys.path,
# robust to whatever working directory Jupyter/VSCode launches the notebook from.
_root = os.getcwd()
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm import tqdm

from src.paths import ISIC_DIR, ISIC_IMAGES_DIR, EDA_ISIC_DIR, PROCESSED_DIR
from src.viz import setup_style, save_fig, CLASS_COLORS

setup_style()

EDA_ISIC_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Back-compat alias for existing plot cells that use f'{OUTPUT_DIR}/...'
OUTPUT_DIR = str(EDA_ISIC_DIR)
print(f"Plots → {OUTPUT_DIR}")
print(f"Processed dataset → {PROCESSED_DIR}")

## Load data

In [ ]:
METADATA_PATH = ISIC_DIR / "metadata.csv"
df = pl.read_csv(METADATA_PATH)
print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} cols from {METADATA_PATH}")

## EDA

### Initial exploration

In [3]:
df.head()

isic_id,attribution,copyright_license,acquisition_day,age_approx,anatom_site_general,anatom_site_special,clin_size_long_diam_mm,concomitant_biopsy,dermoscopic_type,diagnosis_1,diagnosis_2,diagnosis_3,diagnosis_4,diagnosis_5,diagnosis_confirm_type,family_hx_mm,fitzpatrick_skin_type,image_manipulation,image_type,lesion_id,mel_mitotic_index,mel_thick_mm,mel_ulcer,melanocytic,patient_id,personal_hx_mm,pixels_x,pixels_y,rcm_case_id,sex,tbp_tile_type
str,str,str,str,i64,str,str,str,bool,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,bool,str,str,i64,i64,str,str,str
"""ISIC_0000000""","""Anonymous""","""CC-0""",null,55,"""anterior torso""",null,null,false,null,"""Benign""","""Benign melanocytic proliferati…","""Nevus""","""Nevus, Atypical, Dysplastic, o…","""Nevus, Dysplastic""",null,null,null,null,"""dermoscopic""",null,null,null,null,true,null,null,1022,767,null,"""female""",null
"""ISIC_0000001""","""Anonymous""","""CC-0""",null,30,"""anterior torso""",null,null,false,null,"""Benign""","""Benign melanocytic proliferati…","""Nevus""","""Nevus, Atypical, Dysplastic, o…","""Nevus, Dysplastic""",null,null,null,null,"""dermoscopic""",null,null,null,null,true,null,null,1022,767,null,"""female""",null
"""ISIC_0000002""","""Anonymous""","""CC-0""",null,60,"""upper extremity""",null,null,true,null,"""Malignant""","""Malignant melanocytic prolifer…","""Melanoma, NOS""",null,null,"""histopathology""",null,null,null,"""dermoscopic""",null,null,null,null,true,null,null,1022,767,null,"""female""",null
"""ISIC_0000003""","""Anonymous""","""CC-0""",null,30,"""upper extremity""",null,null,false,null,"""Benign""","""Benign melanocytic proliferati…","""Nevus""","""Nevus, Atypical, Dysplastic, o…","""Nevus, Dysplastic""",null,null,null,null,"""dermoscopic""",null,null,null,null,true,null,null,1022,767,null,"""male""",null
"""ISIC_0000004""","""Anonymous""","""CC-0""",null,80,"""posterior torso""",null,null,true,null,"""Malignant""","""Malignant melanocytic prolifer…","""Melanoma, NOS""",null,null,"""histopathology""",null,null,null,"""dermoscopic""",null,null,null,null,true,null,null,1022,767,null,"""male""",null


In [4]:
df.shape

(549590, 32)

In [5]:
df.columns

['isic_id',
 'attribution',
 'copyright_license',
 'acquisition_day',
 'age_approx',
 'anatom_site_general',
 'anatom_site_special',
 'clin_size_long_diam_mm',
 'concomitant_biopsy',
 'dermoscopic_type',
 'diagnosis_1',
 'diagnosis_2',
 'diagnosis_3',
 'diagnosis_4',
 'diagnosis_5',
 'diagnosis_confirm_type',
 'family_hx_mm',
 'fitzpatrick_skin_type',
 'image_manipulation',
 'image_type',
 'lesion_id',
 'mel_mitotic_index',
 'mel_thick_mm',
 'mel_ulcer',
 'melanocytic',
 'patient_id',
 'personal_hx_mm',
 'pixels_x',
 'pixels_y',
 'rcm_case_id',
 'sex',
 'tbp_tile_type']

In [ ]:
interested_columns = [
    'isic_id',
    'lesion_id',
    'patient_id',
    'age_approx',
    'anatom_site_general',
    'anatom_site_special',
    'clin_size_long_diam_mm',
    'fitzpatrick_skin_type',
    'image_type',
    'personal_hx_mm',
    'pixels_x',
    'pixels_y',
    'sex',
    'melanocytic',
    'diagnosis_1',
]

df_selected = df.select(interested_columns)
df_selected.head()

### Define objective target

The classification models use a **2×2 taxonomy** built from two ISIC metadata fields:

| Axis | Source field | Values |
|------|--------------|--------|
| Cell lineage | `melanocytic` (boolean) | `True` → melanocytic, `False` → non-melanocytic |
| Behavior | `diagnosis_1` | `Benign`, `Malignant`, `Indeterminate` |

Crossing them yields the four target classes used downstream:

- **melanocytic-benign** (e.g. nevus) — `melanocytic=True`, `diagnosis_1=Benign`
- **melanocytic-malignant** (e.g. melanoma) — `melanocytic=True`, `diagnosis_1=Malignant`
- **non-melanocytic-benign** (e.g. seborrheic keratosis, dermatofibroma) — `melanocytic=False`, `diagnosis_1=Benign`
- **non-melanocytic-malignant** (e.g. BCC, SCC) — `melanocytic=False`, `diagnosis_1=Malignant`

`Indeterminate` rows and rows where `melanocytic` is null are **excluded** (see the two filter
steps below). The finer diagnosis columns (`diagnosis_2`…`diagnosis_5`) hold the specific lesion
name; the cell below cross-tabulates them so the mapping is auditable rather than assumed.

> Note on naming: the code label form is `Benign-melanocytic` (matching the `CLASS_COLORS` keys in
> `src/viz.py`); it denotes the same class as the thesis's `melanocytic-benign`.

In [ ]:
# Auditable crosswalk: how diagnosis_1 × melanocytic maps to the 4 target classes,
# and which specific diagnoses (diagnosis_2..5) fall under each combination.
crosswalk = (
    df.select(['melanocytic', 'diagnosis_1'])
      .group_by(['melanocytic', 'diagnosis_1'])
      .len()
      .sort('len', descending=True)
)
print("diagnosis_1 × melanocytic (row counts over the full archive):")
print(crosswalk.to_pandas().to_string(index=False))

# Specific lesion names behind each (melanocytic, diagnosis_1) combination.
print("\nExample specific diagnoses per (melanocytic, diagnosis_1):")
for mel_val in [True, False]:
    for dx1 in ['Benign', 'Malignant']:
        names = (
            df.filter(
                (pl.col('melanocytic') == mel_val) & (pl.col('diagnosis_1') == dx1)
            )
            .select('diagnosis_2')
            .drop_nulls()
            .unique()
            .to_series()
            .to_list()
        )
        lineage = 'melanocytic' if mel_val else 'non-melanocytic'
        print(f"  {lineage}-{dx1.lower()}: {sorted(names)[:12]}")

In [7]:
df_selected = df_selected.with_columns(
    pl.when(pl.col('melanocytic').is_not_null() & pl.col('melanocytic'))
      .then(pl.lit('melanocytic'))
      .otherwise(pl.lit('non-melanocytic'))
      .alias('melanocytic_str')
)

df_selected = df_selected.with_columns(
    (pl.col('diagnosis_1') + "-" + pl.col('melanocytic_str').cast(pl.Utf8)).alias('target')
)

df_selected.head()

isic_id,lesion_id,patient_id,age_approx,anatom_site_general,anatom_site_special,clin_size_long_diam_mm,fitzpatrick_skin_type,image_type,personal_hx_mm,sex,melanocytic,diagnosis_1,melanocytic_str,target
str,str,str,i64,str,str,str,str,str,str,str,bool,str,str,str
"""ISIC_0000000""",null,null,55,"""anterior torso""",null,null,null,"""dermoscopic""",null,"""female""",true,"""Benign""","""melanocytic""","""Benign-melanocytic"""
"""ISIC_0000001""",null,null,30,"""anterior torso""",null,null,null,"""dermoscopic""",null,"""female""",true,"""Benign""","""melanocytic""","""Benign-melanocytic"""
"""ISIC_0000002""",null,null,60,"""upper extremity""",null,null,null,"""dermoscopic""",null,"""female""",true,"""Malignant""","""melanocytic""","""Malignant-melanocytic"""
"""ISIC_0000003""",null,null,30,"""upper extremity""",null,null,null,"""dermoscopic""",null,"""male""",true,"""Benign""","""melanocytic""","""Benign-melanocytic"""
"""ISIC_0000004""",null,null,80,"""posterior torso""",null,null,null,"""dermoscopic""",null,"""male""",true,"""Malignant""","""melanocytic""","""Malignant-melanocytic"""


In [8]:
print(df_selected['target'].value_counts())

shape: (7, 2)
┌───────────────────────────────┬────────┐
│ target                        ┆ count  │
│ ---                           ┆ ---    │
│ str                           ┆ u32    │
╞═══════════════════════════════╪════════╡
│ Benign-melanocytic            ┆ 48893  │
│ null                          ┆ 19527  │
│ Benign-non-melanocytic        ┆ 450772 │
│ Malignant-non-melanocytic     ┆ 16465  │
│ Malignant-melanocytic         ┆ 10619  │
│ Indeterminate-melanocytic     ┆ 395    │
│ Indeterminate-non-melanocytic ┆ 2919   │
└───────────────────────────────┴────────┘


The raw `target` above has 7 groups. Only the **4** benign/malignant × melanocytic/non-melanocytic
combinations are kept; the rest are dropped and **must be accounted for** in the thesis:

- `null` (~19.5k) — `diagnosis_1` is missing, so behavior is unknown.
- `Indeterminate-*` (~3.3k) — biopsy could not classify benign vs malignant.

**Important caveat about `Benign-non-melanocytic` (~450k).** The `melanocytic_str` rule labels a row
`non-melanocytic` when `melanocytic` is `True`-and-false **or null**, so the huge 450k count here is
dominated by rows whose `melanocytic` flag is actually *null* (450,045 nulls, see the null-count cell
below). Those rows are removed a few cells later (`melanocytic.is_not_null()`), collapsing this class
to ~9.5k. The retained non-melanocytic rows are therefore only those with an **explicit**
`melanocytic == False`. This is the single biggest data-reduction decision in the pipeline
(~473k of 549k rows discarded) and is quantified explicitly in the imbalance section.

In [9]:
interest_targets = ['Malignant-melanocytic', 'Benign-melanocytic', 'Benign-non-melanocytic', 'Malignant-non-melanocytic']

In [10]:
mask = pl.col('target').is_in(interest_targets)
df_selected = df_selected.filter(mask)

print(df_selected['target'].value_counts())

shape: (4, 2)
┌───────────────────────────┬────────┐
│ target                    ┆ count  │
│ ---                       ┆ ---    │
│ str                       ┆ u32    │
╞═══════════════════════════╪════════╡
│ Malignant-non-melanocytic ┆ 16465  │
│ Malignant-melanocytic     ┆ 10619  │
│ Benign-melanocytic        ┆ 48893  │
│ Benign-non-melanocytic    ┆ 450772 │
└───────────────────────────┴────────┘


### Detect null values

In [11]:
#count missing values per column
missing_counts = df_selected.null_count()
missing_counts

isic_id,lesion_id,patient_id,age_approx,anatom_site_general,anatom_site_special,clin_size_long_diam_mm,fitzpatrick_skin_type,image_type,personal_hx_mm,sex,melanocytic,diagnosis_1,melanocytic_str,target
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,409158,55439,17129,39010,522596,120638,516628,1,508655,26093,450045,0,0,0


In [12]:
df_selected = df_selected.filter(pl.col('melanocytic').is_not_null())

In [ ]:
# Keep: isic_id, lesion_id, patient_id, age_approx, anatom_site_general,
#       pixels_x, pixels_y, sex, target
df_selected = df_selected.drop([
    'anatom_site_special',
    'clin_size_long_diam_mm',
    'fitzpatrick_skin_type',
    'image_type',
    'personal_hx_mm',
    'melanocytic',
    'diagnosis_1',
    'melanocytic_str',
])

In [14]:
df_selected.shape

(76704, 5)

### Class imbalance

The 4 retained classes are strongly imbalanced. For the classification models this drives the choice
of **stratified splitting**, **class weights**, or **resampling**. The cell below reports the per-class
counts, their share, and the imbalance ratio (largest / smallest class).

In [ ]:
# Imbalance summary over the retained 4-class set
cls_counts = (
    df_selected['target'].value_counts()
    .sort('count', descending=True)
    .to_pandas()
)
total = int(cls_counts['count'].sum())
cls_counts['pct'] = (cls_counts['count'] / total * 100).round(2)

n_max = int(cls_counts['count'].max())
n_min = int(cls_counts['count'].min())

print(f"Retained rows: {total:,}")
print(cls_counts.to_string(index=False))
print(f"\nImbalance ratio (majority / minority): {n_max / n_min:.1f}x")
print(f"Suggested inverse-frequency class weights (normalized to mean 1):")
w = total / (len(cls_counts) * cls_counts['count'])
w = (w / w.mean()).round(3)
for cls, wi in zip(cls_counts['target'], w):
    print(f"  {cls:<28} {wi}")

In [15]:
missing_counts = df_selected.null_count()
missing_counts

isic_id,age_approx,anatom_site_general,sex,target
u32,u32,u32,u32,u32
0,14221,29636,14074,0


### Impute age and sex

`age_approx` and `sex` are imputed **per class** with the class-wise mode (not a global mode), so the
filled values stay consistent with each class's demographic profile — e.g. the melanocytic-benign
(nevus) class skews young, so its age mode (~15) differs sharply from the malignant classes (~70–85).

**Caveats for the thesis:** mode imputation collapses variance and can bias the age distribution toward
the most frequent bucket; ~18% of rows have age imputed and ~18% sex. An alternative worth noting is
leaving them missing and letting the classifier handle NaNs, or flagging imputed rows with an indicator
column. The magnitude of imputed values is printed below.

In [16]:
classes = df_selected['target'].unique().to_list()
classes

['Benign-non-melanocytic',
 'Malignant-non-melanocytic',
 'Malignant-melanocytic',
 'Benign-melanocytic']

In [17]:
for class_name in classes:
    dfx = df_selected.filter(pl.col('target') == class_name)
    
    age_mode = dfx['age_approx'].drop_nulls().mode()[0]
    sex_mode = dfx['sex'].drop_nulls().mode()[0]
    
    print(f"Class: {class_name}, Age Mode: {age_mode}, Sex Mode: {sex_mode}")

    df_selected = df_selected.with_columns(
        pl.when(pl.col('target') == class_name)
        .then(pl.col('age_approx').fill_null(age_mode))
        .otherwise(pl.col('age_approx'))
        .alias('age_approx')
    )

    df_selected = df_selected.with_columns(
        pl.when(pl.col('target') == class_name)
        .then(pl.col('sex').fill_null(sex_mode))
        .otherwise(pl.col('sex'))
        .alias('sex')
    )

Class: Benign-non-melanocytic, Age Mode: 70, Sex Mode: male
Class: Malignant-non-melanocytic, Age Mode: 85, Sex Mode: male
Class: Malignant-melanocytic, Age Mode: 70, Sex Mode: male
Class: Benign-melanocytic, Age Mode: 15, Sex Mode: male


In [18]:
missing_counts = df_selected.null_count()
missing_counts

isic_id,age_approx,anatom_site_general,sex,target
u32,u32,u32,u32,u32
0,0,29636,0,0


### Impute a new category for null values in anatom_site_general

In [ ]:
df_selected = df_selected.with_columns(
    pl.col("anatom_site_general").fill_null("unknown")
)

In [20]:
missing_counts = df_selected.null_count()
missing_counts

isic_id,age_approx,anatom_site_general,sex,target
u32,u32,u32,u32,u32
0,0,0,0,0


### Transform anatom_site_general column to one hot encoding format

In [ ]:
# Rename column
df_selected = df_selected.rename({"anatom_site_general": "anatom_site"})

In [22]:
df_selected = df_selected.to_dummies("anatom_site")

In [23]:
df_selected.head()

isic_id,age_approx,anatom_site_anterior torso,anatom_site_head/neck,anatom_site_lateral torso,anatom_site_lower extremity,anatom_site_oral/genital,anatom_site_palms/soles,anatom_site_posterior torso,anatom_site_unknown,anatom_site_upper extremity,sex,target
str,i64,u8,u8,u8,u8,u8,u8,u8,u8,u8,str,str
"""ISIC_0000000""",55,1,0,0,0,0,0,0,0,0,"""female""","""Benign-melanocytic"""
"""ISIC_0000001""",30,1,0,0,0,0,0,0,0,0,"""female""","""Benign-melanocytic"""
"""ISIC_0000002""",60,0,0,0,0,0,0,0,0,1,"""female""","""Malignant-melanocytic"""
"""ISIC_0000003""",30,0,0,0,0,0,0,0,0,1,"""male""","""Benign-melanocytic"""
"""ISIC_0000004""",80,0,0,0,0,0,0,1,0,0,"""male""","""Malignant-melanocytic"""


### Duplicate-image inspection (`lesion_id`, `patient_id`)

Many lesions in ISIC have multiple images and many patients have multiple lesions. If we don't account for that when splitting train/val/test, the same lesion (or even the same patient) can leak across splits and inflate metrics. This cell quantifies the redundancy before we decide a split strategy.

In [ ]:
n_total    = df_selected.shape[0]
n_lesions  = df_selected['lesion_id'].drop_nulls().n_unique()
n_patients = df_selected['patient_id'].drop_nulls().n_unique()

print(f"Total images:           {n_total:,}")
print(f"Unique lesion_id:       {n_lesions:,}  ({n_total / max(n_lesions, 1):.2f} imgs/lesion)")
print(f"Unique patient_id:      {n_patients:,}  ({n_total / max(n_patients, 1):.2f} imgs/patient)")
print(f"Rows with NULL lesion_id:  {df_selected['lesion_id'].null_count():,}")
print(f"Rows with NULL patient_id: {df_selected['patient_id'].null_count():,}")

imgs_per_lesion = (
    df_selected
    .drop_nulls('lesion_id')
    .group_by('lesion_id')
    .len()
    .rename({'len': 'n_imgs'})
)
multi = imgs_per_lesion.filter(pl.col('n_imgs') > 1).shape[0]
print(f"\nLesions with > 1 image: {multi:,}")
print("\nImages-per-lesion distribution:")
print(imgs_per_lesion['n_imgs'].describe())

### Image dimensions per class

Most ISIC images are 640×480 or 1024×768, but custom-uploaded subsets vary widely. Knowing the spread
per class helps decide the input resize/crop strategy for the classifiers.

**Domain-shift note (segmentation transfer).** These images are segmented by a Mask R-CNN trained on
**HAM10000**, which is uniform 600×450 dermoscopic imagery. ISIC spans a much wider resolution range and
mixes acquisition types, so the segmentation model faces a domain shift (resolution, field of view,
possible clinical vs dermoscopic modality). This motivates checking the dimension spread here and, for
the thesis, spot-checking segmentation quality on the higher-variance ISIC subsets rather than assuming
HAM10000 performance transfers directly.

In [ ]:
dim_summary = (
    df_selected
    .drop_nulls(['pixels_x', 'pixels_y'])
    .group_by('target')
    .agg([
        pl.col('isic_id').count().alias('n'),
        pl.col('pixels_x').mean().round(0).alias('mean_w'),
        pl.col('pixels_x').median().alias('median_w'),
        pl.col('pixels_y').mean().round(0).alias('mean_h'),
        pl.col('pixels_y').median().alias('median_h'),
    ])
    .sort('target')
)
print("Image dimensions by class:")
print(dim_summary.to_pandas().to_string(index=False))

# Scatter (sample to keep plot legible)
df_pd = df_selected.drop_nulls(['pixels_x', 'pixels_y']).to_pandas()
sample = df_pd.sample(min(5000, len(df_pd)), random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=sample, x='pixels_x', y='pixels_y', hue='target',
    palette=CLASS_COLORS, alpha=0.4, s=18, ax=ax,
)
ax.set_title('Image dimensions (pixels_x × pixels_y) by class — sample n=5000')
ax.set_xlabel('Width (px)')
ax.set_ylabel('Height (px)')
ax.legend(title='Class', loc='upper right', fontsize=9)
sns.despine(ax=ax)
plt.tight_layout()
save_fig(fig, EDA_ISIC_DIR, '05_image_dimensions')
plt.show()

## Select final image set

`df_final` is the cleaned, imputed set of images that will be (1) segmented by the HAM10000-trained
Mask R-CNN and (2) used to train the 4-class classification models.

In [ ]:
df_final = df_selected
print(f"Final image set (segmentation + classification): {df_final.shape[0]:,}")

In [ ]:
# (disk cross-check and pruning removed — full archive is in the GCS bucket)

In [ ]:
df_final.head()

In [33]:
df_final.head()

isic_id,age_approx,anatom_site_anterior torso,anatom_site_head/neck,anatom_site_lateral torso,anatom_site_lower extremity,anatom_site_oral/genital,anatom_site_palms/soles,anatom_site_posterior torso,anatom_site_unknown,anatom_site_upper extremity,sex,target
str,i64,u8,u8,u8,u8,u8,u8,u8,u8,u8,str,str
"""ISIC_0000000""",55,1,0,0,0,0,0,0,0,0,"""female""","""Benign-melanocytic"""
"""ISIC_0000001""",30,1,0,0,0,0,0,0,0,0,"""female""","""Benign-melanocytic"""
"""ISIC_0000002""",60,0,0,0,0,0,0,0,0,1,"""female""","""Malignant-melanocytic"""
"""ISIC_0000003""",30,0,0,0,0,0,0,0,0,1,"""male""","""Benign-melanocytic"""
"""ISIC_0000004""",80,0,0,0,0,0,0,1,0,0,"""male""","""Malignant-melanocytic"""


## Plot data distribution

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── 1. Class distribution ──────────────────────────────────────────────────────
counts = (
    df_final.to_pandas()['target']
    .value_counts()
    .loc[interest_targets]
)

df_plot = pd.DataFrame({
    'target': [t.replace('-', '\n') for t in interest_targets],
    'count': counts.values,
})

fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
    data=df_plot,
    x='target',
    y='count',
    hue='target',
    edgecolor='white',
    linewidth=1.5,
    width=0.6,
    legend=False,
    ax=ax,
)

for bar, val in zip(ax.patches[:len(counts)], counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(counts.values) * 0.012,
        f'{val:,}',
        ha='center', va='bottom', fontsize=12, fontweight='bold', color='#333333',
    )

ax.set_xlabel('Class / Lesion type', labelpad=10)
ax.set_ylabel('Number of images', labelpad=10)
ax.set_title('Class distribution — ISIC Archive', pad=15)
ax.set_ylim(0, max(counts.values) * 1.15)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
sns.despine(ax=ax)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved → {OUTPUT_DIR}/01_class_distribution.png")

## Demographic statistical analysis

Exploration of **age**, **sex** and **anatomical site** broken down by the 4 selected target classes.

All figures are saved automatically to `results/eda_plots/` at 300 dpi.

### 1. Age distribution by class

In [ ]:
# ── 1. Age distribution ────────────────────────────────────────────────────────
# Convert df_final to pandas once for all demographic plots
df_plot = df_final.to_pandas()
df_plot = df_plot[df_plot['target'].isin(interest_targets)].copy()

x_labels = [t.replace('-', '\n') for t in interest_targets]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left — violin plot
sns.violinplot(
    data=df_plot, x='target', y='age_approx',
    order=interest_targets,
    hue='target',
    inner='quartile',
    linewidth=1,
    ax=axes[0],
)
axes[0].set_xticklabels(x_labels)
axes[0].set_title('Age distribution — Violin')
axes[0].set_xlabel('')
axes[0].set_ylabel('Approximate age (years)')
sns.despine(ax=axes[0])

# Right — box plot
sns.boxplot(
    data=df_plot, x='target', y='age_approx',
    order=interest_targets,
    hue='target',
    flierprops=dict(marker='o', markerfacecolor='gray', markersize=2,
                    alpha=0.25, linestyle='none'),
    linewidth=1.2,
    ax=axes[1],
)
axes[1].set_xticklabels(x_labels)
axes[1].set_title('Age distribution — Boxplot')
axes[1].set_xlabel('')
axes[1].set_ylabel('Approximate age (years)')
sns.despine(ax=axes[1])

fig.suptitle('Age distribution by class — ISIC Archive', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_age_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved → {OUTPUT_DIR}/02_age_distribution.png")

In [ ]:
# ── Age histograms per class ───────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 10))

# unique figure
ax_all = fig.add_subplot(1, 1, 1)
for cls in interest_targets:
    sns.kdeplot(
        data=df_plot[df_plot['target'] == cls],
        x='age_approx',
        color=CLASS_COLORS[cls],
        fill=True,
        alpha=0.30,
        linewidth=2,
        label=cls.replace('-', '\n'),
        ax=ax_all,
    )
ax_all.set_title('Comparison — all classes (density)')
ax_all.set_xlabel('Age (years)', labelpad=8)
ax_all.set_ylabel('Density', labelpad=8)
ax_all.legend(title='Class', fontsize=9, title_fontsize=9,
              loc='upper left', framealpha=0.85)
sns.despine(ax=ax_all)

fig.suptitle('Age histograms by class — ISIC Archive', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02b_age_histograms.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved → {OUTPUT_DIR}/02b_age_histograms.png")

### 2. Sex distribution by class

In [ ]:
# ── 2. Sex distribution ────────────────────────────────────────────────────────
df_sex = (
    df_plot
    .groupby(['target', 'sex'])
    .size()
    .reset_index(name='count')
)
df_sex['pct'] = (
    df_sex.groupby('target')['count'].transform(lambda x: x / x.sum() * 100)
)

#sex_palette = {'male': '#5B9BD5', 'female': '#ED7D31'}
x_labels = [t.replace('-', '\n') for t in interest_targets]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left — absolute counts
sns.barplot(
    data=df_sex, x='target', y='count', hue='sex', order=interest_targets,
    edgecolor='white', linewidth=1, ax=axes[0],
)
axes[0].set_xticklabels(x_labels)
axes[0].set_title('Absolute count by sex')
axes[0].set_xlabel('')
axes[0].set_ylabel('Number of images')
axes[0].legend(title='Sex')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
sns.despine(ax=axes[0])

# Right — proportional
sns.barplot(
    data=df_sex, x='target', y='pct', hue='sex', order=interest_targets,
    edgecolor='white', linewidth=1, ax=axes[1],
)
axes[1].set_xticklabels(x_labels)
axes[1].set_title('Proportion by sex (%)')
axes[1].set_xlabel('')
axes[1].set_ylabel('Percentage (%)')
axes[1].legend(title='Sex')
sns.despine(ax=axes[1])

fig.suptitle('Sex distribution by class — ISIC Archive', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03_sex_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved → {OUTPUT_DIR}/03_sex_distribution.png")

### 3. Anatomical site distribution by class

In [ ]:
# ── 3. Anatomical site distribution ───────────────────────────────────────────
# Reconstruct original anatom_site column from one-hot encoded columns
anatom_cols = [c for c in df_final.columns if c.startswith('anatom_site_')]
df_anatom = df_final.select(['target'] + anatom_cols).to_pandas()
df_anatom['anatom_site'] = (
    df_anatom[anatom_cols]
    .idxmax(axis=1)
    .str.replace('anatom_site_', '', regex=False)
)
df_anatom = df_anatom[df_anatom['target'].isin(interest_targets)]

# Percentage per (class, site) — rows sum to 100 %
heatmap_data = (
    df_anatom
    .groupby(['target', 'anatom_site'])
    .size()
    .reset_index(name='count')
    .assign(total=lambda x: x.groupby('target')['count'].transform('sum'))
    .assign(pct=lambda x: x['count'] / x['total'] * 100)
    .pivot(index='target', columns='anatom_site', values='pct')
    .fillna(0)
    .loc[interest_targets]
)

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.1f',
    cmap='Blues',
    linewidths=0.5,
    linecolor='#e0e0e0',
    cbar_kws={'label': '% within class', 'shrink': 0.8},
    ax=ax,
    vmin=0,
)
ax.set_title('Anatomical site distribution by class (%)', pad=15)
ax.set_xlabel('Anatomical site', labelpad=10)
ax.set_ylabel('Class', labelpad=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, ha='right')
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/04_anatom_site_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved → {OUTPUT_DIR}/04_anatom_site_distribution.png")

### 4. Descriptive age statistics by class

In [ ]:
# ── 4. Descriptive statistics — Age per class ─────────────────────────────────
stats = (
    df_plot.groupby('target')['age_approx']
    .describe(percentiles=[0.25, 0.5, 0.75])
    .round(1)
    .loc[interest_targets]
)
stats.index.name = 'Class'
stats.columns = ['N', 'Mean', 'Std. dev.', 'Min.', 'Q1 (25%)', 'Median (50%)', 'Q3 (75%)', 'Max.']
stats['N'] = stats['N'].astype(int)

print("Descriptive age statistics by class:")
display(stats)

### 5. Sample images per class

Qualitative look at a few images from each of the 4 target classes — a sanity check on image content
(dermoscopic vs clinical, artifacts, framing) before the segmentation + classification stages. Requires
the ISIC images to be available at `ISIC_IMAGES_DIR` (GCS mount); missing files are reported, not skipped
silently.

In [ ]:
from pathlib import Path
from PIL import Image

n_per_class = 3
img_dir = Path(ISIC_IMAGES_DIR)

fig, axes = plt.subplots(len(interest_targets), n_per_class,
                         figsize=(4 * n_per_class, 4 * len(interest_targets)))
fig.suptitle('Sample images per class — ISIC Archive', y=1.0)

missing = 0
for row, cls in enumerate(interest_targets):
    ids = (
        df_final.filter(pl.col('target') == cls)
        .select('isic_id')
        .to_series()
        .to_list()
    )
    # take the first n found on disk
    shown = 0
    for isic_id in ids:
        if shown >= n_per_class:
            break
        img_path = img_dir / f"{isic_id}.jpg"
        ax = axes[row, shown]
        if not img_path.exists():
            missing += 1
            continue
        ax.imshow(Image.open(img_path).convert('RGB'))
        ax.set_title(f"{cls}\n{isic_id}", fontsize=8)
        ax.axis('off')
        shown += 1
    # blank any unfilled columns in this row
    for c in range(shown, n_per_class):
        axes[row, c].axis('off')
    if shown == 0:
        axes[row, 0].text(0.5, 0.5, f"No images found for\n{cls}",
                          ha='center', va='center', fontsize=10)

if missing:
    print(f"WARNING: {missing} sampled image file(s) not found under {img_dir}")

plt.tight_layout()
save_fig(fig, EDA_ISIC_DIR, '06_samples_per_class')
plt.show()

## Save processed dataset

The filtered, imputed, one-hot-encoded `df_final` is saved as a Parquet under `results/processed/` so downstream notebooks (feature extraction, classifier training) can load it directly without re-running the EDA pipeline.

In [ ]:
processed_path = PROCESSED_DIR / "isic_selected.parquet"
df_final.write_parquet(processed_path)

ids_path = PROCESSED_DIR / "isic_selected_ids.csv"
df_final.select("isic_id").write_csv(ids_path)

print(f"Processed dataset → {processed_path}  ({df_final.shape[0]:,} rows)")
print(f"Image IDs list    → {ids_path}")
print(f"  cols: {df_final.columns}")